# Directional-Corridor Transformer-PPO v2 — One-Click Runner

Upload this notebook to Google Colab and choose **Runtime → Run all**. The notebook mounts Google Drive, installs the verified SUMO environment, updates the repository, and resumes the pipeline from its persisted state.

If the `training` stage is already complete, the saved champion is reused: the notebook runs metric-v2 revalidation, comparison, and result generation **without retraining**. Do not delete the Drive directory `G11project-directional-corridor-v2`.

In [ ]:
from pathlib import Path

from google.colab import drive

DRIVE_ROOT = Path("/content/drive")
if not (DRIVE_ROOT / "MyDrive").is_dir():
    drive.mount(str(DRIVE_ROOT))
else:
    print("Google Drive is already mounted.")

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY = Path("/content/G11project")
REPOSITORY_URL = "https://github.com/yangyu-rgb/G11project.git"

if REPOSITORY.exists() and not (REPOSITORY / ".git").is_dir():
    raise RuntimeError(
        f"{REPOSITORY} exists but is not a Git repository. "
        "Restart the Colab runtime and use Runtime → Run all again."
    )

if not REPOSITORY.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPOSITORY)],
        check=True,
    )
else:
    tracked_changes = subprocess.run(
        ["git", "-C", str(REPOSITORY), "status", "--porcelain", "--untracked-files=no"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    if tracked_changes:
        raise RuntimeError(
            "The temporary Colab repository contains modified tracked files. "
            "Restart the runtime instead of overwriting them."
        )
    subprocess.run(
        ["git", "-C", str(REPOSITORY), "pull", "--ff-only", "origin", "main"],
        check=True,
    )

os.chdir(REPOSITORY)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", "./BackEnd[dev]"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "eclipse-sumo==1.27.1"],
    check=True,
)

import sumo  # noqa: E402

SUMO_HOME = Path(sumo.SUMO_HOME).resolve()
SUMO_BINARY = SUMO_HOME / "bin/sumo"
if not SUMO_BINARY.is_file():
    raise RuntimeError(f"SUMO executable is missing under {SUMO_HOME}")
os.environ["SUMO_HOME"] = str(SUMO_HOME)
os.environ["PATH"] = str(SUMO_HOME / "bin") + os.pathsep + os.environ.get("PATH", "")
os.environ["PYTHONPATH"] = str(SUMO_HOME / "tools") + os.pathsep + os.environ.get("PYTHONPATH", "")
subprocess.run([str(SUMO_BINARY), "--version"], check=True)
subprocess.run(
    [sys.executable, str(REPOSITORY / "BackEnd/scripts/verify_sumo.py")],
    cwd=REPOSITORY,
    check=True,
)
commit = subprocess.run(
    ["git", "-C", str(REPOSITORY), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print("Repository commit:", commit)
print("SUMO_HOME:", SUMO_HOME)

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import torch

REPOSITORY = Path("/content/G11project")
PERSISTENT_ROOT = Path("/content/drive/MyDrive/G11project-directional-corridor-v2")
WORK_ROOT = Path("/content/g11-directional-v2-work")
PIPELINE = REPOSITORY / "BackEnd/scripts/run_adaptive_demo_pipeline.py"
CONFIG = REPOSITORY / "BackEnd/configs/adaptive_demo_training.yaml"
STATE_PATH = PERSISTENT_ROOT / "adaptive_demo_state.json"
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. This is acceptable only when training is already complete.")
print("Persistent output:", PERSISTENT_ROOT)

base_command = [
    sys.executable,
    str(PIPELINE),
    "--persistent-root",
    str(PERSISTENT_ROOT),
    "--work-root",
    str(WORK_ROOT),
    "--config",
    str(CONFIG),
    "--time-budget-minutes",
    "270",
]


def read_status() -> dict:
    completed = subprocess.run(
        base_command + ["--status"],
        cwd=REPOSITORY,
        check=True,
        capture_output=True,
        text=True,
    )
    return json.loads(completed.stdout)


initial_status = read_status()
print("Initial pipeline status:")
print(json.dumps(initial_status, indent=2, ensure_ascii=False))
if initial_status["stages"]["training"]["status"] == "completed":
    print("Training is already complete. Reusing the saved champion; no retraining will occur.")
elif not torch.cuda.is_available():
    raise RuntimeError(
        "Training is not complete and no GPU is available. Select a GPU runtime, then Run all."
    )

paused = False
while True:
    status = read_status()
    stage = status["next_stage"]
    if stage is None:
        break
    print(f"\n===== Starting/resuming stage: {stage} =====")
    process = subprocess.Popen(
        base_command + ["--stage", stage],
        cwd=REPOSITORY,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError("Unable to capture pipeline output.")
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code == 75:
        paused = True
        print("The stage was checkpointed safely. Rerun this cell to resume from Drive.")
        break
    if return_code != 0:
        if STATE_PATH.is_file():
            print("\nSaved failure state:")
            print(STATE_PATH.read_text(encoding="utf-8"))
        raise RuntimeError(f"Stage {stage!r} failed with return code {return_code}.")

final_status = read_status()
print("\nFinal pipeline status:")
print(json.dumps(final_status, indent=2, ensure_ascii=False))
if paused:
    print("Pipeline paused safely; the saved champion and progress remain in Google Drive.")

In [ ]:
import json
from pathlib import Path

PERSISTENT_ROOT = Path("/content/drive/MyDrive/G11project-directional-corridor-v2")
REVALIDATION = PERSISTENT_ROOT / "champion/revalidation.json"
SUMMARY = PERSISTENT_ROOT / "comparison_results/summary.json"
MANIFEST = PERSISTENT_ROOT / "champion/model_manifest.json"
PRESENTATION_RESULTS = PERSISTENT_ROOT / "presentation_results"

if REVALIDATION.is_file():
    revalidation = json.loads(REVALIDATION.read_text(encoding="utf-8"))
    print("\n=== Champion Revalidation ===")
    print(json.dumps(revalidation, indent=2, ensure_ascii=False))
else:
    print("Revalidation output is not available yet.")

if SUMMARY.is_file():
    summary = json.loads(SUMMARY.read_text(encoding="utf-8"))
    behavior = summary.get("behavioral_gate", {})
    behavior_overview = {
        key: behavior.get(key)
        for key in (
            "passed",
            "event_count",
            "unique_incident_count",
            "action_signature_count",
            "receiver_signature_count",
            "policy_seed_consistent",
            "forward_notifications",
            "nearest_follower_opportunities",
            "nearest_follower_coverage",
            "receiver_signature_unique_ratio",
            "context_adaptive_action",
        )
        if key in behavior
    }
    print("\n=== Final Acceptance ===")
    print(json.dumps(summary.get("acceptance", {}), indent=2, ensure_ascii=False))
    print("\n=== Directional-Behavior Evidence ===")
    print(json.dumps(behavior_overview, indent=2, ensure_ascii=False))
    print("\n=== Coverage Eligibility ===")
    print(json.dumps(summary.get("coverage_eligibility", {}), indent=2, ensure_ascii=False))
else:
    print("Comparison summary is not available yet.")

print("\n=== Final Artifacts ===")
print("Model manifest:", MANIFEST, "ready =", MANIFEST.is_file())
print("Comparison summary:", SUMMARY, "ready =", SUMMARY.is_file())
print("Presentation results:", PRESENTATION_RESULTS, "ready =", PRESENTATION_RESULTS.is_dir())
if MANIFEST.is_file():
    print("\nSUCCESS: the accepted model manifest and final presentation outputs are ready.")
else:
    print(
        "\nThe manifest is not ready. Read the failed acceptance checks above; do not use old results."
    )